# 📍 Assign NR Index and Coordinates from Database Lookup (Option B: DB Master Overwrite)

This standalone Jupyter notebook assigns **NR Indexes** and populates **Coordinates** for any separate processed file (`.xlsx`, `.csv`, or `.parquet`) without modifying existing pipeline code.

### 🔑 Key Features:
1. **Existing DB Lookup & Master Overwrite (Option B)**:
   - Finds existing `internal_index_id` from `public.transactions` for matching `(location_name, project_name)`.
   - **Option B Enabled**: Always overwrites the file's project coordinates with the DB master coordinates (`project_latitude`, `project_longitude`), ensuring 100% consistency with the database.
2. **Sequential NR Assignment for New Projects**:
   - Queries PostgreSQL for the highest existing `nr` index for the target city (`SELECT MAX(...)`) and assigns new `nr` numbers starting from `highest_nr + 1`.
3. **Consistency across Rows**:
   - All rows with the same `(location_name, project_name)` receive the exact same index and coordinates.
4. **Location Coordinates Lookup**:
   - Populates `location_latitude` and `location_longitude` from `public.dim_location`.
5. **Safe & Non-Destructive**:
   - Zero modifications to existing codebase files.

In [1]:
# Step 1: Imports & Setup
import os
import re
import sys
import time
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import psycopg2

warnings.filterwarnings("ignore")

# Styling helpers for console
CYAN = "\033[96m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
BOLD = "\033[1m"
RESET = "\033[0m"

print(f"{GREEN}✓ Libraries loaded successfully.{RESET}")


✓ Libraries loaded successfully.


In [2]:
# Step 2: Configuration & Parameters
# ============================================================
# 1. FILE CONFIGURATION
# Set the path to your separate processed file (.xlsx, .csv, or .parquet)
# ============================================================
INPUT_FILE_PATH = r"E:\Nilesh\IGR_processing_System\vadgaon_sheri_db1.xlsx"

# Output file path (leave None to auto-generate <filename>_indexed_coords.xlsx)
OUTPUT_FILE_PATH = None

# ============================================================
# 2. CITY SELECTION
# City ID mappings: 9 = Pune, 8 = Mumbai, 12 = Thane, 15 = Dubai
# ============================================================
CITY_ID = 9
CITY_NAME = "Pune"

# ============================================================
# 3. DATABASE CONFIGURATION
# ============================================================
DB_PARAMS = {
    "host": "localhost",
    "port": "5432",
    "database": "nilesh",
    "user": "postgres",
    "password": "nilesh",
}

# ============================================================
# 4. PROCESSING OPTIONS (OPTION B)
# ============================================================
# Option B: Overwrite file project coords with DB master coords for matched projects
OVERWRITE_COORDS_FROM_DB = True

# Unify coordinates across all rows of the same (project + location) within the file
UNIFY_IN_FILE_PROJECT_COORDS = True

# Populate location latitude & longitude from public.dim_location
POPULATE_LOCATION_COORDS = True

# Enrich project coordinates from DB transactions
POPULATE_PROJECT_COORDS_FROM_DB = True

# Optional manual override for NR starting number (leave None to auto-query DB)
MANUAL_START_NR = None

print(f"{BOLD}Configuration Summary:{RESET}")
print(f"  Target City:               {CYAN}{CITY_NAME} (City ID: {CITY_ID}){RESET}")
print(f"  Database:                  {DB_PARAMS['database']} @ {DB_PARAMS['host']}:{DB_PARAMS['port']}")
print(f"  Input File:                {INPUT_FILE_PATH}")
print(f"  Overwrite Coords from DB:  {GREEN}{OVERWRITE_COORDS_FROM_DB} (Option B enabled){RESET}")
print(f"  Unify Coords in File:      {GREEN}{UNIFY_IN_FILE_PROJECT_COORDS}{RESET}")


Configuration Summary:
  Target City:               Pune (City ID: 9)
  Database:                  nilesh @ localhost:5432
  Input File:                E:\Nilesh\IGR_processing_System\vadgaon_sheri_db1.xlsx
  Overwrite Coords from DB:  True (Option B enabled)
  Unify Coords in File:      True


In [3]:
# Step 3: Helper Functions - Database Check & File Loader
def test_db_connection(params: dict) -> bool:
    """Verifies PostgreSQL connectivity."""
    try:
        conn = psycopg2.connect(**params)
        with conn.cursor() as cur:
            cur.execute("SELECT 1;")
        conn.close()
        print(f"{GREEN}✓ PostgreSQL connection successful.{RESET}")
        return True
    except Exception as e:
        print(f"{RED}✗ PostgreSQL connection failed: {e}{RESET}")
        return False

def create_sample_file_if_missing(file_path: str):
    """Creates a sample Excel file if the configured input file does not exist yet."""
    if not os.path.exists(file_path):
        sample_data = {
            "project_name": ["Amanora Park Town", "Blue Ridge", "Unknown Sample Heights", "Unknown Sample Heights", "New Horizon", "Amanora Park Town"],
            "location_name": ["hadapsar", "hinjewadi", "wakad", "wakad", "baner", "hadapsar"],
            "index": [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
            "project_latitude": [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
            "project_longitude": [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan],
            "agreement_price": [8500000, 7200000, 6100000, 6150000, 9500000, 8900000],
        }
        sample_df = pd.DataFrame(sample_data)
        sample_df.to_excel(file_path, index=False)
        print(f"{YELLOW}ℹ Created sample test file for demonstration:{RESET} {file_path}")

def load_data_file(file_path: str) -> pd.DataFrame:
    """Loads a DataFrame from .xlsx, .xls, .csv, or .parquet file."""
    if not os.path.exists(file_path):
        create_sample_file_if_missing(file_path)
    
    ext = os.path.splitext(file_path)[1].lower()
    print(f"Loading input file: {os.path.basename(file_path)} ...")
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    elif ext == ".csv":
        df = pd.read_csv(file_path, low_memory=False)
    elif ext == ".parquet":
        df = pd.read_parquet(file_path)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")
    
    print(f"{GREEN}✓ Loaded {len(df):,} rows and {len(df.columns)} columns.{RESET}")
    return df

# Check DB connection
test_db_connection(DB_PARAMS)


✓ PostgreSQL connection successful.


True

In [4]:
# Step 4: Query Database for Highest NR Index
def get_highest_nr_from_db(city_id: int, db_params: dict, manual_start: int = None) -> int:
    """
    Queries public.transactions for the highest 'nr' index for the city.
    """
    if manual_start is not None and manual_start > 0:
        print(f"  {YELLOW}ℹ Using manual override starting NR: nr{manual_start}{RESET}")
        return manual_start - 1

    print(f"Querying highest NR in database for City ID {city_id} ...")
    t0 = time.time()
    conn = psycopg2.connect(**db_params)
    cur = conn.cursor()
    cur.execute("""
        SELECT MAX(
            (regexp_match(internal_index_id, '^nr(\\d+)', 'i'))[1]::int
        )
        FROM public.transactions
        WHERE city_id = %s AND internal_index_id ~* '^nr\\d+';
    """, (city_id,))
    max_nr = cur.fetchone()[0] or 0
    cur.close()
    conn.close()
    elapsed = round(time.time() - t0, 2)
    
    print(f"{GREEN}✓ Completed in {elapsed}s.{RESET}")
    print(f"  Highest existing NR in DB:      {BOLD}nr{max_nr}{RESET}")
    print(f"  Brand new NR sequence starts at: {BOLD}nr{max_nr + 1}{RESET}")
    return max_nr


In [5]:
# Step 5: Query Database for Existing Project Mappings
def fetch_db_project_mappings(city_id: int, db_params: dict) -> tuple[dict, dict]:
    """
    Fetches (lower(location_name), lower(project_name)) -> internal_index_id and (latitude, longitude)
    from public.transactions for the target city.
    """
    def _to_float(v):
        if v is None or pd.isna(v):
            return None
        try:
            val = float(v)
            return val if not np.isnan(val) else None
        except (ValueError, TypeError):
            return None

    db_mapping = {}
    db_coords_mapping = {}
    print(f"Fetching existing project mappings from DB for City ID {city_id} ...")
    t0 = time.time()
    try:
        conn = psycopg2.connect(**db_params)
        with conn.cursor() as cur:
            cur.execute("""
                SELECT lower(trim(location_name)), lower(trim(project_name)), internal_index_id, project_latitude, project_longitude
                FROM public.transactions
                WHERE city_id = %s
                  AND project_name IS NOT NULL AND location_name IS NOT NULL;
            """, (city_id,))
            for loc, proj, idx, lat, lon in cur.fetchall():
                if loc and proj:
                    key = (loc, proj)
                    if idx and str(idx).strip() and key not in db_mapping:
                        db_mapping[key] = str(idx).strip()
                    plat, plon = _to_float(lat), _to_float(lon)
                    if plat is not None and plon is not None and key not in db_coords_mapping:
                        db_coords_mapping[key] = (plat, plon)
        conn.close()
        elapsed = round(time.time() - t0, 2)
        print(f"{GREEN}✓ Loaded {len(db_mapping):,} project index mappings and {len(db_coords_mapping):,} coordinate pairs in {elapsed}s.{RESET}")
    except Exception as e:
        print(f"{YELLOW}⚠️ DB index mapping lookup note: {e}{RESET}")

    return db_mapping, db_coords_mapping


In [6]:
# Step 6: Core Logic - Assign NR Indexes & Enrich Project Coordinates (Option B)
def assign_nr_indexes(
    df: pd.DataFrame,
    target_city_id: int,
    db_params: dict,
    manual_start_nr: int = None,
    populate_coords: bool = True,
    overwrite_coords: bool = True,
    unify_coords: bool = True
) -> tuple[pd.DataFrame, dict]:
    """
    Assigns index and coordinates:
    1. Reuses existing index for matching (location_name, project_name) from current file or DB.
    2. Assigns new sequential NR index starting from (highest DB NR + 1) for new project + location pairs.
    3. Option B (overwrite_coords=True): Overwrites file coordinates with DB master coordinates for matched projects.
    4. Unifies coordinates across all rows of the same (project + location) in the file.
    """
    df = df.copy()

    # 1. Normalize index & project columns
    if "index" not in df.columns:
        if "internal_index_id" in df.columns:
            df["index"] = df["internal_index_id"]
        else:
            df["index"] = np.nan

    df["index"] = df["index"].astype(object).replace(r"^\s*$", np.nan, regex=True)

    if "project_name" in df.columns:
        df["project_name"] = df["project_name"].replace(r"^\s*$", np.nan, regex=True)

    if "project_latitude" not in df.columns:
        df["project_latitude"] = np.nan
    else:
        df["project_latitude"] = pd.to_numeric(df["project_latitude"], errors="coerce")

    if "project_longitude" not in df.columns:
        df["project_longitude"] = np.nan
    else:
        df["project_longitude"] = pd.to_numeric(df["project_longitude"], errors="coerce")

    def _to_float(v):
        if v is None or pd.isna(v):
            return None
        try:
            val = float(v)
            return val if not np.isnan(val) else None
        except (ValueError, TypeError):
            return None

    loc_col = next((c for c in ["location_name", "registered_document_village_name", "location"] if c in df.columns), None)

    # 2. Fetch DB mappings & Max NR
    db_mapping, db_coords_mapping = fetch_db_project_mappings(target_city_id, db_params)
    max_nr = get_highest_nr_from_db(target_city_id, db_params, manual_start=manual_start_nr)
    next_num = max_nr + 1

    # 3. Build current file mapping for (location_name, project_name) -> index & coordinates
    file_mapping = {}
    file_coords_mapping = {}
    if loc_col and "project_name" in df.columns:
        valid_rows = df.dropna(subset=[loc_col, "project_name"])
        for _, row in valid_rows.iterrows():
            l_key = str(row[loc_col]).strip().lower()
            p_key = str(row["project_name"]).strip().lower()
            if l_key and p_key:
                k = (l_key, p_key)
                if pd.notna(row.get("index")) and str(row["index"]).strip():
                    file_mapping.setdefault(k, str(row["index"]).strip())
                plat, plon = _to_float(row.get("project_latitude")), _to_float(row.get("project_longitude"))
                if plat is not None and plon is not None:
                    file_coords_mapping.setdefault(k, (plat, plon))

    existing_indexes = set(df["index"].dropna().astype(str).str.strip().str.lower())

    # 4. Filter eligible rows (index is missing & project_name present)
    if "project_name" in df.columns:
        rows_to_assign = df.index[
            df["index"].isna()
            & df["project_name"].notna()
            & df["project_name"].astype("string").str.strip().ne("")
        ]
    else:
        rows_to_assign = df.index[df["index"].isna()]

    print(f"\n{CYAN}Rows eligible for NR / Index assignment:{RESET} {len(rows_to_assign):,}")

    new_count = 0
    reused_file_count = 0
    reused_db_count = 0

    # 5. Assign Indexes
    for i in rows_to_assign:
        proj = str(df.at[i, "project_name"]).strip().lower() if "project_name" in df.columns and pd.notna(df.at[i, "project_name"]) else ""
        loc = str(df.at[i, loc_col]).strip().lower() if loc_col and pd.notna(df.at[i, loc_col]) else ""
        key = (loc, proj) if (loc and proj) else None

        if key and key in file_mapping:
            df.at[i, "index"] = file_mapping[key]
            reused_file_count += 1
        elif key and key in db_mapping:
            existing_idx = db_mapping[key]
            df.at[i, "index"] = existing_idx
            file_mapping[key] = existing_idx
            existing_indexes.add(str(existing_idx).strip().lower())
            reused_db_count += 1
        else:
            while f"nr{next_num}" in existing_indexes:
                next_num += 1
            new_index = f"nr{next_num}"
            df.at[i, "index"] = new_index
            if key:
                file_mapping[key] = new_index
                db_mapping[key] = new_index
            existing_indexes.add(new_index.lower())
            next_num += 1
            new_count += 1

    # 6. Populate & Overwrite project coordinates (Option B)
    coords_reused = 0
    coords_overwritten_from_db = 0
    coords_unified_in_file = 0

    if populate_coords and loc_col and "project_name" in df.columns:
        for i in df.index:
            p_val = df.at[i, "project_name"]
            l_val = df.at[i, loc_col]
            if pd.notna(p_val) and pd.notna(l_val):
                p_key = str(p_val).strip().lower()
                l_key = str(l_val).strip().lower()
                if p_key and l_key:
                    k = (l_key, p_key)
                    db_coords = db_coords_mapping.get(k)
                    file_coords = file_coords_mapping.get(k)

                    curr_lat = _to_float(df.at[i, "project_latitude"])
                    curr_lon = _to_float(df.at[i, "project_longitude"])
                    lat_blank = curr_lat is None
                    lon_blank = curr_lon is None

                    # Case 1 (Option B): Project found in DB -> DB is Master, overwrite file coords
                    if overwrite_coords and db_coords:
                        d_lat, d_lon = db_coords
                        if lat_blank or lon_blank:
                            df.at[i, "project_latitude"] = float(d_lat)
                            df.at[i, "project_longitude"] = float(d_lon)
                            coords_reused += 1
                        elif abs(curr_lat - d_lat) > 1e-6 or abs(curr_lon - d_lon) > 1e-6:
                            df.at[i, "project_latitude"] = float(d_lat)
                            df.at[i, "project_longitude"] = float(d_lon)
                            coords_overwritten_from_db += 1

                    # Case 2: Coordinates blank -> fill from file or DB
                    elif lat_blank or lon_blank:
                        coords = file_coords or db_coords
                        if coords:
                            c_lat, c_lon = coords
                            if lat_blank and c_lat is not None:
                                df.at[i, "project_latitude"] = float(c_lat)
                            if lon_blank and c_lon is not None:
                                df.at[i, "project_longitude"] = float(c_lon)
                            coords_reused += 1

                    # Case 3: Unify across file rows for non-DB projects
                    elif unify_coords and file_coords and not db_coords:
                        f_lat, f_lon = file_coords
                        if abs(curr_lat - f_lat) > 1e-6 or abs(curr_lon - f_lon) > 1e-6:
                            df.at[i, "project_latitude"] = float(f_lat)
                            df.at[i, "project_longitude"] = float(f_lon)
                            coords_unified_in_file += 1

    stats = {
        "highest_db_nr": max_nr,
        "eligible_rows": len(rows_to_assign),
        "reused_from_file": reused_file_count,
        "reused_from_db": reused_db_count,
        "total_reused": reused_file_count + reused_db_count,
        "brand_new_nr_assigned": new_count,
        "highest_new_nr": f"nr{next_num - 1}" if new_count > 0 else f"nr{max_nr}",
        "blank_remaining": int(df["index"].isna().sum()),
        "coords_reused": coords_reused,
        "coords_overwritten_from_db": coords_overwritten_from_db,
        "coords_unified_in_file": coords_unified_in_file,
    }

    print(f"  {GREEN}✓ Reused existing Index (In-File):{RESET} {reused_file_count:,}")
    print(f"  {GREEN}✓ Reused existing Index (DB Lookup):{RESET} {reused_db_count:,}")
    print(f"  {GREEN}✓ Brand new NR assigned:{RESET} {new_count:,}")
    if new_count > 0:
        print(f"  {CYAN}Sequence start:{RESET} nr{max_nr + 1}  ->  {CYAN}Highest new NR:{RESET} nr{next_num - 1}")
    print(f"  {GREEN}✓ Project coords filled for blank rows:{RESET} {coords_reused:,}")
    if overwrite_coords:
        print(f"  {GREEN}✓ Project coords overwritten with DB Master (Option B):{RESET} {coords_overwritten_from_db:,} rows")
    if unify_coords and coords_unified_in_file > 0:
        print(f"  {GREEN}✓ Project coords unified across in-file rows:{RESET} {coords_unified_in_file:,} rows")
    print(f"  {YELLOW}Blank indexes remaining:{RESET} {stats['blank_remaining']:,}")

    return df, stats


In [21]:
# Step 7: Location Coordinates Lookup from public.dim_location
def populate_location_coords(df: pd.DataFrame, city_id: int, db_params: dict) -> pd.DataFrame:
    """
    Populates location_latitude and location_longitude from public.dim_location for location_name.
    Exact function from project.py / pipeline_core.py.
    """
    loc_col = next((c for c in ["location_name", "registered_document_village_name", "location"] if c in df.columns), None)
    if not loc_col:
        return df

    df = df.copy()
    if "location_latitude" not in df.columns:
        df["location_latitude"] = np.nan
    if "location_longitude" not in df.columns:
        df["location_longitude"] = np.nan

    print(f"Populating location coordinates from public.dim_location for City ID {city_id} ...")
    try:
        conn = psycopg2.connect(**db_params)
        coords = pd.read_sql_query(
            "SELECT DISTINCT LOWER(TRIM(location_name)) AS loc, location_latitude AS latitude, location_longitude AS longitude "
            "FROM public.dim_location WHERE city_id = %s AND location_latitude IS NOT NULL;",
            conn,
            params=(city_id,)
        )
        conn.close()
        coords = coords.drop_duplicates(subset=["loc"]).set_index("loc")
        keys = df[loc_col].astype(str).str.strip().str.lower()

        missing_mask = df["location_latitude"].isna() | df["location_longitude"].isna()
        df.loc[missing_mask, "location_latitude"] = df.loc[missing_mask, "location_latitude"].fillna(keys.map(coords["latitude"]))
        df.loc[missing_mask, "location_longitude"] = df.loc[missing_mask, "location_longitude"].fillna(keys.map(coords["longitude"]))
        
        loc_filled = int(df["location_latitude"].notna().sum())
        print(f"  {GREEN}✓ Location LatLong populated ({loc_filled:,} non-null coordinates total).{RESET}")
    except Exception as e:
        print(f"  {YELLOW}⚠️ Location coordinates lookup note: {e}{RESET}")

    return df


In [7]:
# Step 8: Execute Pipeline on Target File
print(f"{BOLD}--- Starting Processing (Option B: DB Master Overwrite) ---{RESET}")

# 1. Load Data
df = load_data_file(INPUT_FILE_PATH)

# 2. Run NR Index & Coordinates Assignment (with Option B overwrite)
df_processed, stats = assign_nr_indexes(
    df=df,
    target_city_id=CITY_ID,
    db_params=DB_PARAMS,
    manual_start_nr=MANUAL_START_NR,
    populate_coords=POPULATE_PROJECT_COORDS_FROM_DB,
    overwrite_coords=OVERWRITE_COORDS_FROM_DB,
    unify_coords=UNIFY_IN_FILE_PROJECT_COORDS
)

# 3. Populate Location Coordinates (from dim_location)
if POPULATE_LOCATION_COORDS:
    df_processed = populate_location_coords(df_processed, CITY_ID, DB_PARAMS)

print(f"\n{GREEN}{BOLD}✓ Index and Coordinates processing completed successfully!{RESET}")


--- Starting Processing (Option B: DB Master Overwrite) ---
Loading input file: vadgaon_sheri_db1.xlsx ...
✓ Loaded 7,202 rows and 221 columns.
Fetching existing project mappings from DB for City ID 9 ...
✓ Loaded 57,622 project index mappings and 52,511 coordinate pairs in 119.63s.
Querying highest NR in database for City ID 9 ...
✓ Completed in 104.75s.
  Highest existing NR in DB:      nr51269
  Brand new NR sequence starts at: nr51270

Rows eligible for NR / Index assignment: 4,798
  ✓ Reused existing Index (In-File): 3,640
  ✓ Reused existing Index (DB Lookup): 176
  ✓ Brand new NR assigned: 982
  Sequence start: nr51270  ->  Highest new NR: nr52251
  ✓ Project coords filled for blank rows: 0
  ✓ Project coords overwritten with DB Master (Option B): 2,055 rows
  Blank indexes remaining: 961


NameError: name 'populate_location_coords' is not defined

In [23]:
# Step 9: Review Summary Statistics & Preview Data
summary_records = [
    ("Target City", f"{CITY_NAME} (City ID: {CITY_ID})"),
    ("Total Rows", f"{len(df_processed):,}"),
    ("Eligible Rows for Indexing", f"{stats['eligible_rows']:,}"),
    ("Reused Existing Index (DB Lookup)", f"{stats['reused_from_db']:,}"),
    ("Reused Existing Index (In-File)", f"{stats['reused_from_file']:,}"),
    ("Brand New NR Assigned", f"{stats['brand_new_nr_assigned']:,}"),
    ("Highest Existing DB NR", f"nr{stats['highest_db_nr']}"),
    ("Sequence Start for New NR", f"nr{stats['highest_db_nr'] + 1}"),
    ("Highest New NR Assigned", f"{stats['highest_new_nr']}"),
    ("Project Coords Filled (Blank Rows)", f"{stats['coords_reused']:,}"),
    ("Project Coords Overwritten (DB Master - Option B)", f"{stats['coords_overwritten_from_db']:,}"),
    ("Project Coords Unified (In-File Rows)", f"{stats['coords_unified_in_file']:,}"),
    ("Remaining Blank Indexes", f"{stats['blank_remaining']:,}"),
]

summary_df = pd.DataFrame(summary_records, columns=["Metric", "Value"])
print("\n=================== SUMMARY REPORT ===================")
try:
    display(summary_df)
except NameError:
    print(summary_df.to_string(index=False))

# Preview key columns
preview_columns = [c for c in ["index", "project_name", "location_name", "project_latitude", "project_longitude", "location_latitude", "location_longitude", "agreement_price"] if c in df_processed.columns]
print("\n=================== SAMPLE PROCESSED DATA ===================")
try:
    display(df_processed[preview_columns].head(20))
except NameError:
    print(df_processed[preview_columns].head(20).to_string())



=================== SUMMARY REPORT ===================


,Metric,Value
0,Target City,Pune (City ID: 9)
1,Total Rows,"24,036"
2,Eligible Rows for Indexing,"18,087"
3,Reused Existing Index (DB Lookup),756
4,Reused Existing Index (In-File),"14,974"
5,Brand New NR Assigned,"2,357"
6,Highest Existing DB NR,nr48895
7,Sequence Start for New NR,nr48896
8,Highest New NR Assigned,nr51252
9,Project Coords Filled (Blank Rows),6



=================== SAMPLE PROCESSED DATA ===================


,index,project_name,location_name,project_latitude,project_longitude,location_latitude,location_longitude,agreement_price
0,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
1,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
2,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
3,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,4342000.0
4,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,5500000.0
5,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
6,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
7,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
8,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,0.0
9,197.0,38 Divine,Kondhwa Khurd,18.466262,73.889524,18.470052,73.885986,5716001.0


In [8]:
# Step 10: Save Resulting Processed File
if not OUTPUT_FILE_PATH:
    base, ext = os.path.splitext(INPUT_FILE_PATH)
    OUTPUT_FILE_PATH = f"{base}_indexed_coords{ext if ext else '.xlsx'}"

print(f"Writing output file to: {OUTPUT_FILE_PATH} ...")
out_ext = os.path.splitext(OUTPUT_FILE_PATH)[1].lower()
if out_ext in [".xlsx", ".xls"]:
    df_processed.to_excel(OUTPUT_FILE_PATH, index=False)
elif out_ext == ".csv":
    df_processed.to_csv(OUTPUT_FILE_PATH, index=False)
elif out_ext == ".parquet":
    df_processed.to_parquet(OUTPUT_FILE_PATH, index=False)
else:
    df_processed.to_excel(OUTPUT_FILE_PATH + ".xlsx", index=False)
    OUTPUT_FILE_PATH += ".xlsx"

out_size_mb = os.path.getsize(OUTPUT_FILE_PATH) / (1024 * 1024)
print(f"{GREEN}✓ File saved successfully!{RESET}")
print(f"  Location: {OUTPUT_FILE_PATH}")
print(f"  Size:     {out_size_mb:.2f} MB")


Writing output file to: E:\Nilesh\IGR_processing_System\vadgaon_sheri_db1_indexed_coords.xlsx ...
✓ File saved successfully!
  Location: E:\Nilesh\IGR_processing_System\vadgaon_sheri_db1_indexed_coords.xlsx
  Size:     5.57 MB


In [25]:
# Step 11 (Optional): Google Places API Geocoding for Remaining Unmapped Coordinates
# Set RUN_GOOGLE_GEOCODING to True if you wish to run Google Places API for missing coordinates
RUN_GOOGLE_GEOCODING = False

if RUN_GOOGLE_GEOCODING:
    try:
        from project_coordinates import populate_project_coordinates
        print("Running Google Places API geocoding for unmapped projects...")
        df_processed = populate_project_coordinates(df_processed)
        print(f"{GREEN}✓ Google geocoding complete.{RESET}")
        # Re-save with geocoded coordinates
        if out_ext in [".xlsx", ".xls"]:
            df_processed.to_excel(OUTPUT_FILE_PATH, index=False)
        elif out_ext == ".csv":
            df_processed.to_csv(OUTPUT_FILE_PATH, index=False)
        elif out_ext == ".parquet":
            df_processed.to_parquet(OUTPUT_FILE_PATH, index=False)
        print(f"{GREEN}✓ Updated file with Google coordinates saved to {OUTPUT_FILE_PATH}{RESET}")
    except ImportError:
        print(f"{YELLOW}⚠️ project_coordinates.py not found in current directory. Geocoding skipped.{RESET}")
    except Exception as e:
        print(f"{RED}✗ Geocoding error: {e}{RESET}")
else:
    print("Google Places geocoding skipped (RUN_GOOGLE_GEOCODING = False).")


Google Places geocoding skipped (RUN_GOOGLE_GEOCODING = False).
